# 🧭 Notebook 3: Acoustic Phase Interferometry & Direction of Arrival (AoA)

Welcome to the **Acoustic Direction of Arrival (AoA) Laboratory**.

This notebook guides you step-by-step through measuring the physical angle of an incident sound wave using a **two-microphone phase interferometer** on the **PYNQ-Z2 board**.

---

### 🏛️ Physical Operating Principle
When a sound wave arrives at an angle $\theta$ relative to array broadside ($0^\circ$):
* The path difference between the two microphones separated by distance $d$ is $\Delta r = d \cdot \sin(\theta)$.
* This spatial path difference produces an acoustic phase delay $\Delta \phi$:
$$\Delta \phi = \frac{2\pi f_0}{c(T)} \cdot d \cdot \sin(\theta)$$
* Inverting this equation gives the incident bearing angle $\theta$:
$$\theta = \arcsin\left(\frac{c(T) \cdot \Delta \phi}{2\pi f_0 \cdot d}\right)$$

> **Spatial Aliasing Constraint:** To prevent phase wrapping ambiguity, the microphone spacing $d$ must satisfy $d \le \frac{\lambda}{2} = \frac{c}{2 f_0}$. For our $2610\,\text{Hz}$ buzzer, $d \le 6.58\,\text{cm}$. We use a baseline of **$d = 5.0\,\text{cm}$**.

## 1. Hardware Initialization & Profile Loading

Connect your setup:
1. **Mic 1** $\to$ Arduino **A0** (`Vaux1`).
2. **Mic 2** $\to$ Arduino **A1** (`Vaux9`).
3. Place microphone capsules side-by-side with center-to-center spacing **$d = 5.0\,\text{cm}$**.
4. Power your buzzer circuit ($2\text{N}2222\text{A}$ switch with control wire plugged into $3.3\,\text{V}$ for steady tone).

In [ ]:
import json
from pathlib import Path
import numpy as np
from pynq_localizer import MicrophoneArrayOverlay

# 1. Initialize Hardware Overlay (50 kSPS dual continuous streaming)
ol = MicrophoneArrayOverlay()

# 2. Physical Array Parameters
mic_distance_m = 0.05  # 5.0 cm baseline
profile_path = Path("profiles/active_buzzer_2610hz.json")

if profile_path.exists():
    with open(profile_path, "r", encoding="utf-8") as f:
        p_data = json.load(f)
    f0 = p_data.get("f_res_hz", 2609.73)
    d_max = p_data.get("max_mic_spacing_aoa_cm", 6.58)
else:
    f0 = 2609.73
    d_max = 6.58

print(f"✅ Hardware Overlay Active: {ol.fs_per_ch:.0f} SPS per channel")
print(f"✅ Target Carrier Frequency : f0 = {f0:.2f} Hz")
print(f"✅ Microphone Baseline     : d  = {mic_distance_m*100:.1f} cm (Aliasing Limit <= {d_max:.2f} cm)")

## 2. Quick 3-Position Sanity Check

Before marking an entire protractor, test the 3 cardinal positions:
* **Center ($0^\circ$):** Hold buzzer directly in front $\approx 30\,\text{cm}$ away.
* **Right ($+45^\circ$):** Hold buzzer to the right (closer to Mic 2 / A1).
* **Left ($-45^\circ$):** Hold buzzer to the left (closer to Mic 1 / A0).

In [ ]:
positions = [
    ("CENTER (Directly in front)", "≈ 0.0°"),
    ("RIGHT (Closer to Mic 2)", "Positive (+30° to +45°)"),
    ("LEFT (Closer to Mic 1)", "Negative (-30° to -45°)")
]

for label, expected in positions:
    input(f"👉 Place buzzer at {label} (~30 cm away) and press [Enter]...")
    
    res = ol.capture_aoa_frame(
        f_target=f0,
        mic_distance_m=mic_distance_m,
        noise_gate_v=0.010
    )
    
    print(f"   • Measured Angle : {res['theta_deg']:+5.1f}° (Expected: {expected})")
    print(f"   • Phase Shift Δφ : {res['delta_phi_rad']:+5.3f} rad ({np.degrees(res['delta_phi_rad']):+5.1f}°)")
    print(f"   • Wave Coherence : {res['coherence']:.3f} | Status: {res['status']}\n")

## 3. Multi-Station Protractor Benchmark

Place a protractor or mark radial lines at $r \approx 30\,\text{cm}$ on your desk for:
$$\theta_{\text{target}} \in \{-45^\circ, -30^\circ, 0^\circ, +30^\circ, +45^\circ\}$$

The cell below records **$N=20$ frames per station** to compute mean bearing $\bar{\theta}$ and standard deviation $\sigma_\theta$.

In [ ]:
import time

test_stations_deg = [-45.0, -30.0, 0.0, +30.0, +45.0]
n_bursts = 20
results = {}

print("🚀 Starting Guided Protractor Loop...")

for target_deg in test_stations_deg:
    input(f"\n👉 Place buzzer at {target_deg:+5.1f}° mark and press [Enter]...")
    
    measured_angles = []
    measured_coherences = []
    
    for _ in range(n_bursts):
        frame = ol.capture_aoa_frame(
            f_target=f0,
            mic_distance_m=mic_distance_m,
            noise_gate_v=0.010,
            timeout=0.5
        )
        if frame["status"] == "ACTIVE_VALID":
            measured_angles.append(frame["theta_deg"])
            measured_coherences.append(frame["coherence"])
        time.sleep(0.01)
        
    mean_th = float(np.mean(measured_angles))
    std_th = float(np.std(measured_angles))
    err = abs(mean_th - target_deg)
    
    results[str(target_deg)] = {
        "target_deg": target_deg,
        "measured_mean_deg": mean_th,
        "measured_std_deg": std_th,
        "abs_error_deg": err,
        "coherence": float(np.mean(measured_coherences))
    }
    
    tag = "✅" if err <= 3.0 else "⚠️"
    print(f"   {tag} Target: {target_deg:+5.1f}° ──► Measured: {mean_th:+5.2f}° ± {std_th:4.2f}° (Error = {err:4.2f}°)")

## 4. Angular Linearity & Quality Gate Evaluation

Evaluate whether your physical interferometry setup meets the **$\le 3.0^\circ$ accuracy gate** across the central $[-45^\circ, +45^\circ]$ sector.

In [ ]:
import plotly.graph_objects as go

# Summary Table Printout
print(f"{'Target Station':<16} | {'Measured Mean':<16} | {'Std Dev (σ)':<12} | {'Abs Error':<10} | {'Status'}")
print("-" * 72)

targets = [d["target_deg"] for d in results.values()]
means = [d["measured_mean_deg"] for d in results.values()]
stds = [d["measured_std_deg"] for d in results.values()]
errors = [d["abs_error_deg"] for d in results.values()]

for d in results.values():
    passed = d["abs_error_deg"] <= 3.0
    print(f"{d['target_deg']:+5.1f}°{'':<10} | {d['measured_mean_deg']:+5.2f}°{'':<10} | ±{d['measured_std_deg']:4.2f}°{'':<6} | {d['abs_error_deg']:4.2f}°{'':<5} | {'✅ PASS' if passed else '❌ FAIL'}")

print("-" * 72)
print(f"Mean Absolute Error across all stations: {np.mean(errors):.2f}°\n")

# Linearity Calibration Plot with ±3.0° Gate Corridor
x_ideal = np.linspace(-60, 60, 100)
fig_lin = go.Figure()
fig_lin.add_scatter(x=x_ideal, y=x_ideal + 3.0, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), showlegend=False)
fig_lin.add_scatter(x=x_ideal, y=x_ideal - 3.0, mode="lines", line=dict(width=0.5, color="rgba(0, 255, 204, 0.2)", dash="dot"), fill="tonexty", fillcolor="rgba(0, 255, 204, 0.12)", name="±3.0° Quality Corridor")
fig_lin.add_scatter(x=x_ideal, y=x_ideal, mode="lines", line=dict(color="gray", dash="dash"), name="Ideal (y = x)")
fig_lin.add_scatter(x=targets, y=means, error_y=dict(type="data", array=stds, visible=True), mode="markers", marker=dict(size=10, color="#00FFCC"), name="Measured Stations")

fig_lin.update_layout(template="plotly_dark", height=450, title="<b>Angular Linearity: Measured vs. Target Protractor Angle</b>")
fig_lin.update_xaxes(title="Target Angle (°)", range=[-60, 60])
fig_lin.update_yaxes(title="Measured Angle (°)", range=[-60, 60])
fig_lin.show()

## 5. 2D Spatial Acoustic Beam Reconstruction

Reconstruct the physical acoustic ray vectors in 2D Euclidean space originating from the microphone baseline ($d = 5.0\,\text{cm}$) and pointing toward the measured buzzer locations.

In [ ]:
r_beam = 0.30  # 30 cm distance
colors = ["#FFA500", "#00E5FF", "#76FF03", "#E040FB", "#FFD600"]

fig_rays = go.Figure()
# Draw physical microphone baseline
fig_rays.add_scatter(x=[-0.025, 0.025], y=[0, 0], mode="lines+markers", marker=dict(size=8, color=["#00FFCC", "#FF007F"]), line=dict(color="white", width=3), name="Mic Baseline (d=5cm)")

for idx, (t_tgt, t_meas) in enumerate(zip(targets, means)):
    c = colors[idx % len(colors)]
    # Target direction (dashed line)
    rad_t = np.radians(t_tgt)
    fig_rays.add_scatter(x=[0, r_beam * np.sin(rad_t)], y=[0, r_beam * np.cos(rad_t)], mode="lines", line=dict(color=c, dash="dot", width=1.5), showlegend=False)
    # Measured direction (solid ray)
    rad_m = np.radians(t_meas)
    fig_rays.add_scatter(x=[0, r_beam * np.sin(rad_m)], y=[0, r_beam * np.cos(rad_m)], mode="lines+markers", line=dict(color=c, width=2.5), marker=dict(size=[0, 7]), name=f"Ray {t_tgt:+.0f}°")

fig_rays.update_layout(template="plotly_dark", height=450, title="<b>2D Reconstructed Directional Acoustic Beams</b>")
fig_rays.update_xaxes(title="Horizontal Position X (m)", range=[-0.25, 0.25])
fig_rays.update_yaxes(title="Forward Distance Y (m)", range=[-0.02, 0.35])
fig_rays.show()

## 6. Live 100 Hz Streaming Dashboard

Launch the interactive dashboard. Click **Tab 4 (🧭 Direction of Arrival)** and wave the buzzer left and right in front of the microphones to observe the real-time bearing trajectory $\theta(t)$.

In [ ]:
# Launch the real-time 4-tab dashboard with AoA enabled
app = ol.kinematics_dashboard(
    window_duration_sec=10.0,
    hop_ms=10.0,
    aoa_mic_distance_m=mic_distance_m
)

## 7. Clean Hardware Teardown

When finished, stop the dashboard threads and release the FPGA DMA memory buffers.

In [ ]:
if 'app' in locals():
    app.stop()
ol.close()
print("🔒 FPGA hardware and DMA buffers cleanly released.")